# LFM2.5 — Basic Usage

## Imports

In [1]:
from pprint import pprint

import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("MPS (Apple Silicon GPU) available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

torch: 2.13.0
transformers: 5.14.1
MPS (Apple Silicon GPU) available: True
CUDA available: False


## Load Model and Tokenizer

In [2]:
MODEL_ID = "LiquidAI/LFM2.5-350M"

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

print(f"Architecture: {model.config.architectures}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")

config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.49k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  709MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Architecture: ['Lfm2ForCausalLM']
Parameters: 354,483,968
Device: mps:0
Dtype: torch.bfloat16


## Single Turn Generation

In [3]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|startoftext|><|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



In [4]:
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
pprint(inputs, sort_dicts=False, width=120)

{'input_ids': tensor([[    1,     6,  6423,   708,  3493,   856,   779,  5706,   803,  4481,
           540,     7,   708,     6, 64015,   708]], device='mps:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}


In [5]:
outputs = model.generate(**inputs, max_new_tokens=128)
print(outputs)

tensor([[    1,     6,  6423,   708,  3493,   856,   779,  5706,   803,  4481,
           540,     7,   708,     6, 64015,   708,  1098,  5706,   803,  4481,
           856,  5242,   523,     7]], device='mps:0')


In [6]:
print(tokenizer.decode(outputs[0]))

<|startoftext|><|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>


In [7]:
input_len = inputs["input_ids"].shape[-1]
response = tokenizer.decode(outputs[0][input_len:])
print(response)

The capital of France is Paris.<|im_end|>


In [8]:
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print(response)

The capital of France is Paris.


## System Prompt

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who responds in all capitals."},
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|startoftext|><|im_start|>system
You are a helpful assistant who responds in all capitals.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



In [10]:
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]
outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

THE CAPITAL OF FRANCE IS PARIS.


## Multi-Turn Generation

In [11]:
messages = [
    {"role": "user", "content": "What's the capital of France?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    return_dict=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

The capital of France is Paris.


In [12]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of France is Paris.'}]


In [13]:
prompt = "What is a famous landmark there?"

messages.append({"role": "user", "content": prompt})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of France is Paris.'},
 {'role': 'user', 'content': 'What is a famous landmark there?'}]


In [14]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    return_dict=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

A famous landmark in Paris is the Eiffel Tower. It was built for the 1889 World's Fair and is one of the most recognizable structures in the world.


## Streaming Generation

In [15]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    return_dict=True,
).to(model.device)

streamer = transformers.TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
outputs = model.generate(**inputs, max_new_tokens=128, streamer=streamer)

The capital of France is Paris.
